In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,                
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "figure.dpi": 150,                   
    "grid.alpha": 0.4,                    
})


# Stability analysis of a single fluid

In [2]:

def extract_stable_branch(input_folder, output_folder, file_pattern="*.csv", 
                          pressure_col='Central_Pressure_MeV_fm3', mass_col='Mass_sol'):
    """
    Reads CSV files, isolates the stable branch (up to maximum mass), 
    ensures ascending pressure, and saves cleaned CSVs.
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Grab all matching files in the input folder
    search_pattern = os.path.join(input_folder, file_pattern)
    filepaths = glob.glob(search_pattern)
    
    if not filepaths:
        print(f"No files found matching: {search_pattern}")
        return
        
    for path in filepaths:
        filename = os.path.basename(path)
        
        try:
            df = pd.read_csv(path)
            
            # Verify necessary columns exist
            if pressure_col not in df.columns or mass_col not in df.columns:
                print(f"Skipping {filename}: Missing '{pressure_col}' or '{mass_col}'.")
                continue
            
            # 1. Sort strictly by ascending central pressure and reset the index
            df = df.sort_values(by=pressure_col, ascending=True).reset_index(drop=True)
            
            # 2. Find the exact index of the maximum mass
            max_mass_idx = df[mass_col].idxmax()
            
            # 3. Slice the DataFrame from the beginning up to the max mass
            # .loc[] is inclusive of the end index
            df_stable = df.loc[:max_mass_idx].copy()
            
            # Optional: Add an explicit stability flag to match your other scripts
            df_stable['is_stable'] = True 
            
            # 4. Save the cleaned DataFrame to the output folder
            clean_filename = filename.replace('.csv', '_Cleaned.csv')
            output_path = os.path.join(output_folder, clean_filename)
            
            df_stable.to_csv(output_path, index=False)
            
            max_mass = df_stable[mass_col].max()
            print(f"Processed {filename} -> Saved {len(df_stable)} stable points (Max Mass: {max_mass:.3f})")
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")

# ==========================================
# Example Usage:
# ==========================================
extract_stable_branch(
     input_folder='../data/mr_library', 
     output_folder='../data/mr_library/clean_single_fluids',
     file_pattern="mr_*.csv",
     #pressure_col='p_c',       # Change this if your pressure column is named differently
     #mass_col='Mass_sol'       # Change this if your mass column is named differently
 )

Processed mr_bosonicDM_n_4_mb_300_B_148.csv -> Saved 130 stable points (Max Mass: 2.225)
Processed mr_quark_matter_real.csv -> Saved 166 stable points (Max Mass: 1.854)
Processed mr_bosonicDM_n_40_mb_200_B_148.csv -> Saved 150 stable points (Max Mass: 3.427)
Processed mr_bosonicDM_n_40_mb_700_B_148.csv -> Saved 150 stable points (Max Mass: 0.280)
Processed mr_bosonicDM_n_4_mb_500_B_148.csv -> Saved 130 stable points (Max Mass: 0.801)
Processed mr_bosonicDM_n_40_mb_1000_B_148.csv -> Saved 150 stable points (Max Mass: 0.137)
Processed mr_bosonicDM_n_40_mb_100_B_148.csv -> Saved 150 stable points (Max Mass: 13.709)
Processed mr_bosonicDM_n_4_mb_1000_B_148.csv -> Saved 130 stable points (Max Mass: 0.200)
Processed mr_bosonicDM_n_4_mb_700_B_148.csv -> Saved 130 stable points (Max Mass: 0.409)
Processed mr_bosonicDM_n_40_mb_2000_B_148.csv -> Saved 150 stable points (Max Mass: 0.034)
Processed mr_bosonicDM_n_4_mb_2000_B_148.csv -> Saved 130 stable points (Max Mass: 0.050)
Processed mr_bosonic